In [1]:
import pandas as pd

df = pd.read_parquet("train-00000-of-00001.parquet")

print("Dataset loaded successfully!")
print(df.head())
print(df.columns)
print("Number of rows:", len(df))

Dataset loaded successfully!
                                                text
0                                                   
1                     = Valkyria Chronicles III = \n
2                                                   
3   Senjō no Valkyria 3 : Unrecorded Chronicles (...
4   The game began development in 2010 , carrying...
Index(['text'], dtype='object')
Number of rows: 36718


In [2]:
import pandas as pd
import re
import numpy as np

from collections import Counter



# 2. Load WikiText-2 Dataset


df = pd.read_parquet(
    "train-00000-of-00001.parquet"
)

print("Dataset loaded successfully!")
print("Number of rows:", len(df))
print("Columns:", df.columns.tolist())



# 3. Remove Empty Text


df = df.dropna(subset=['text'])

df = df[df['text'].str.strip() != '']

print("\nRows after removing empty text:", len(df))



# 4. Clean and Tokenize Text


tokens = []

for text in df['text']:

    # Convert to lowercase
    text = text.lower()

    # Remove special characters and numbers
    text = re.sub(
        r'[^a-z\s]',
        ' ',
        text
    )

    # Remove extra spaces
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    # Tokenize
    words = text.split()

    # Add words to token list
    tokens.extend(words)


print("\nText cleaning completed!")

print("Total tokens:", len(tokens))

print("\nFirst 30 tokens:")
print(tokens[:30])



# 5. Build Unigram


unigram = Counter(tokens)

print("\n============================================")
print("UNIGRAM")
print("============================================")

print("Unique words:", len(unigram))

print("\nTop 10 words:")

for word, count in unigram.most_common(10):

    print(word, ":", count)



# 6. Build Bigram

bigram = Counter()

for i in range(len(tokens) - 1):

    pair = (
        tokens[i],
        tokens[i + 1]
    )

    bigram[pair] += 1


print("\n============================================")
print("BIGRAM")
print("============================================")

print("Unique bigrams:", len(bigram))

print("\nTop 10 bigrams:")

for pair, count in bigram.most_common(10):

    print(pair, ":", count)



# 7. Build Trigram


trigram = Counter()

for i in range(len(tokens) - 2):

    triple = (
        tokens[i],
        tokens[i + 1],
        tokens[i + 2]
    )

    trigram[triple] += 1


print("\n============================================")
print("TRIGRAM")
print("============================================")

print("Unique trigrams:", len(trigram))

print("\nTop 10 trigrams:")

for triple, count in trigram.most_common(10):

    print(triple, ":", count)



# 8. Bigram Probability

def bigram_probability(word1, word2):

    pair_count = bigram[
        (word1, word2)
    ]

    word_count = unigram[word1]

    if word_count == 0:

        return 0

    return pair_count / word_count



# 9. Trigram Probability


def trigram_probability(
    word1,
    word2,
    word3
):

    triple_count = trigram[
        (word1, word2, word3)
    ]

    pair_count = bigram[
        (word1, word2)
    ]

    if pair_count == 0:

        return 0

    return triple_count / pair_count



# 10. Predict Next Words


def predict_next_words(
    sentence,
    top_n=5
):

    # Convert to lowercase
    sentence = sentence.lower()

    # Remove unwanted characters
    sentence = re.sub(
        r'[^a-z\s]',
        ' ',
        sentence
    )

    # Remove extra spaces
    sentence = re.sub(
        r'\s+',
        ' ',
        sentence
    ).strip()

    # Tokenize
    words = sentence.split()

    if len(words) == 0:

        return []


    candidates = []


   
    # Use Trigram
  

    if len(words) >= 2:

        word1 = words[-2]
        word2 = words[-1]

        for (
            w1,
            w2,
            w3
        ), count in trigram.items():

            if (
                w1 == word1
                and
                w2 == word2
            ):

                probability = (
                    count /
                    bigram[(word1, word2)]
                )

                candidates.append(
                    (
                        w3,
                        probability,
                        count
                    )
                )


  
    # If no trigram, use Bigram
   
    if len(candidates) == 0:

        last_word = words[-1]

        for (
            w1,
            w2
        ), count in bigram.items():

            if w1 == last_word:

                probability = (
                    count /
                    unigram[last_word]
                )

                candidates.append(
                    (
                        w2,
                        probability,
                        count
                    )
                )


    
    # Sort by probability
    
    candidates.sort(
        key=lambda x: (
            x[1],
            x[2]
        ),
        reverse=True
    )


    
    # Get Top N
    
    final_predictions = []

    seen = set()

    for word, probability, count in candidates:

        if word not in seen:

            final_predictions.append(
                (
                    word,
                    probability
                )
            )

            seen.add(word)

        if len(final_predictions) == top_n:

            break


    return final_predictions



# 11. Accept User Input


print("\n============================================")
print("SMART NEXT-WORD PREDICTOR")
print("============================================")

query = input(
    "\nEnter a sentence or partial sentence: "
)



# 12. Predict

predictions = predict_next_words(
    query,
    top_n=5
)



# 13. Display Results


print("\nInput:")
print(query)

print("\nTop 5 Predicted Next Words:")

if len(predictions) == 0:

    print("No prediction found.")

else:

    for i, (
        word,
        probability
    ) in enumerate(
        predictions,
        start=1
    ):

        print(
            i,
            ".",
            word,
            "- Probability:",
            round(probability, 3)
        )



# 14. Test Multiple Sentences

test_sentences = [
    "the united",
    "machine learning",
    "the first",
    "one of the",
    "in the"
]


print("\n============================================")
print("MULTIPLE TEST CASES")
print("============================================")


for sentence in test_sentences:

    predictions = predict_next_words(
        sentence,
        top_n=5
    )

    print("\nInput:", sentence)

    if len(predictions) == 0:

        print("No prediction found.")

    else:

        for i, (
            word,
            probability
        ) in enumerate(
            predictions,
            start=1
        ):

            print(
                i,
                ".",
                word,
                "-",
                round(probability, 3)
            )


print("\n============================================")
print("PROCESS COMPLETED SUCCESSFULLY!")
print("============================================")

Dataset loaded successfully!
Number of rows: 36718
Columns: ['text']

Rows after removing empty text: 23767

Text cleaning completed!
Total tokens: 1694394

First 30 tokens:
['valkyria', 'chronicles', 'iii', 'senj', 'no', 'valkyria', 'unrecorded', 'chronicles', 'japanese', 'lit', 'valkyria', 'of', 'the', 'battlefield', 'commonly', 'referred', 'to', 'as', 'valkyria', 'chronicles', 'iii', 'outside', 'japan', 'is', 'a', 'tactical', 'role', 'playing', 'video', 'game']

UNIGRAM
Unique words: 61884

Top 10 words:
the : 130771
of : 57032
and : 50738
in : 45027
to : 39524
a : 36834
was : 21008
s : 16375
on : 15158
as : 15063

BIGRAM
Unique bigrams: 667981

Top 10 bigrams:
('of', 'the') : 17476
('in', 'the') : 12801
('to', 'the') : 6082
('on', 'the') : 4523
('and', 'the') : 4458
('for', 'the') : 3741
('at', 'the') : 3240
('from', 'the') : 3027
('by', 'the') : 3005
('as', 'a') : 2924

TRIGRAM
Unique trigrams: 1307606

Top 10 trigrams:
('one', 'of', 'the') : 869
('the', 'united', 'states') : 673



Enter a sentence or partial sentence:  the united



Input:
the united

Top 5 Predicted Next Words:
1 . states - Probability: 0.764
2 . kingdom - Probability: 0.178
3 . nations - Probability: 0.043
4 . arab - Probability: 0.005
5 . confederate - Probability: 0.001

MULTIPLE TEST CASES

Input: the united
1 . states - 0.764
2 . kingdom - 0.178
3 . nations - 0.043
4 . arab - 0.005
5 . confederate - 0.001

Input: machine learning
1 . that - 0.133
2 . curve - 0.093
3 . the - 0.08
4 . to - 0.08
5 . about - 0.067

Input: the first
1 . time - 0.105
2 . of - 0.029
3 . to - 0.027
4 . two - 0.024
5 . game - 0.022

Input: one of the
1 . th - 0.016
2 . season - 0.016
3 . year - 0.012
4 . first - 0.011
5 . game - 0.01

Input: in the
1 . united - 0.031
2 . s - 0.027
3 . th - 0.019
4 . first - 0.018
5 . early - 0.015

PROCESS COMPLETED SUCCESSFULLY!
